# Duka Electronics Sales Exploration (Jan–Jun 2026)

## Executive Summary
* Across 1,000 orders between 2026-01-01 and 2026-06-29, Duka Electronics generated KES 71,792,500 in total revenue.
* Nairobi leads all 5 branches with KES 26,106,100 (36.36% of total revenue) across 376 orders, while Eldoret sits fifth with KES 8,844,200 across 117 orders.
* Laptops generate the highest product revenue at KES 41,535,000 (57.85% of revenue) despite ranking second in volume with 639 units behind Tablets (691 units).
* April is the strongest revenue month at KES 13,518,300 and June is the lowest at KES 10,903,400, a difference of +23.98% (or -19.34% from April to June).
* Filtering for Premium online orders in Nairobi and Mombasa yields 102 orders totaling KES 11,163,000, exported to `outputs/premium_online_coast_capital.csv`.

## First-Five-Minutes Inspection
* **Shape & columns:** `duka_sales.csv` has **1,000 rows** and **6 raw columns** (`order_id`, `order_date`, `branch`, `product`, `quantity`, `channel`, plus `unit_price` = 7 columns), with zero missing values across any column.
* **Data types:** `order_date` is parsed as `datetime64[ns]`; `order_id`, `quantity`, and `unit_price` are `int64`; `branch`, `product`, and `channel` are `object` (strings).
* **Date range:** Orders span **2026-01-01 to 2026-06-29** (180 days across January–June 2026).
* **Key categories:** 5 branches (`Nairobi`, `Mombasa`, `Kisumu`, `Nakuru`, `Eldoret`), 5 products (`Phone` KES 15,000; `Laptop` KES 65,000; `Headphones` KES 2,500; `Tablet` KES 28,000; `Charger` KES 1,200), and 3 channels (`In-store`, `Online`, `Phone order`).

Load Data & Derived Columns

In [1]:
import os
import numpy as np
import pandas as pd

os.makedirs("outputs", exist_ok=True)
os.makedirs("excel", exist_ok=True)


def generate_duka_sales() -> pd.DataFrame:
  rng = np.random.default_rng(2026)
  n = 1000
  products = {
      "Phone": 15000,
      "Laptop": 65000,
      "Headphones": 2500,
      "Tablet": 28000,
      "Charger": 1200,
  }

  sales = pd.DataFrame({
      "order_id": range(1001, 1001 + n),
      "order_date": pd.to_datetime("2026-01-01")
      + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
      "branch": rng.choice(
          ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
          n,
          p=[0.4, 0.2, 0.15, 0.15, 0.1],
      ),
      "product": rng.choice(list(products), n),
      "quantity": rng.integers(1, 6, n),
      "channel": rng.choice(
          ["In-store", "Online", "Phone order"],
          n,
          p=[0.55, 0.35, 0.10],
      ),
  })
  sales["unit_price"] = sales["product"].map(products)
  sales.to_csv("duka_sales.csv", index=False)
  return sales


if not os.path.exists("duka_sales.csv"):
  generate_duka_sales()

df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print(df.columns.tolist())

['order_id', 'order_date', 'branch', 'product', 'quantity', 'channel', 'unit_price']


Task 3: Branch Rankings by Revenue and Orders

In [2]:
if "revenue" not in df.columns:
  df["revenue"] = df["quantity"] * df["unit_price"]

branch_rank = (
    df.groupby("branch")
    .agg(
        total_revenue=("revenue", "sum"),
        order_count=("order_id", "count"),
        avg_order_value=("revenue", "mean"),
    )
    .sort_values("total_revenue", ascending=False)
)
branch_rank["revenue_rank"] = (
    branch_rank["total_revenue"]
    .rank(ascending=False, method="min")
    .astype(int)
)
branch_rank["order_rank"] = (
    branch_rank["order_count"]
    .rank(ascending=False, method="min")
    .astype(int)
)
branch_rank["revenue_share_pct"] = (
    branch_rank["total_revenue"] / df["revenue"].sum() * 100
).round(2)
branch_rank

,total_revenue,order_count,avg_order_value,revenue_rank,order_rank,revenue_share_pct
branch,,,,,,
Nairobi,26106100,376,69431.117021,1,1,36.36
Mombasa,14003700,186,75288.709677,2,2,19.51
Nakuru,11970400,162,73891.358025,3,3,16.67
Kisumu,10868100,159,68352.830189,4,4,15.14
Eldoret,8844200,117,75591.452991,5,5,12.32


### Task 3 Answer: Branch Rankings
**Yes, the revenue ranking and order-count ranking are identical (1st to 5th):**
1. **Nairobi:** KES 26,106,100 (36.36% share) across **376 orders** (Rank 1 in both)
2. **Mombasa:** KES 14,003,700 (19.51% share) across **186 orders** (Rank 2 in both)
3. **Nakuru:** KES 11,970,400 (16.67% share) across **162 orders** (Rank 3 in both)
4. **Kisumu:** KES 10,868,100 (15.14% share) across **159 orders** (Rank 4 in both)
5. **Eldoret:** KES 8,844,200 (12.32% share) across **117 orders** (Rank 5 in both)

Because order volume follows the branch weights (`0.40`, `0.20`, `0.15`, `0.15`, `0.10`) and product/quantity mixes are similar across branches, higher order counts translate directly into higher total revenue.

Task 4: Top Three Products by Revenue and Units Sold

In [3]:
product_summary = (
    df.groupby("product")
    .agg(
        unit_price=("unit_price", "first"),
        total_revenue=("revenue", "sum"),
        units_sold=("quantity", "sum"),
        orders=("order_id", "count"),
    )
)

top3_rev = product_summary.sort_values("total_revenue", ascending=False).head(3)
top3_units = product_summary.sort_values("units_sold", ascending=False).head(3)

print("Top 3 by Revenue:\n", top3_rev[["unit_price", "total_revenue", "units_sold"]])
print("\nTop 3 by Units Sold:\n", top3_units[["unit_price", "units_sold", "total_revenue"]])

Top 3 by Revenue:
          unit_price  total_revenue  units_sold
product                                       
Laptop        65000       41535000         639
Tablet        28000       19348000         691
Phone         15000        8880000         592

Top 3 by Units Sold:
          unit_price  units_sold  total_revenue
product                                       
Tablet        28000         691       19348000
Laptop        65000         639       41535000
Phone         15000         592        8880000


### Task 4 Answer: Top Three Products by Revenue vs. Units Sold
* **Top 3 by Revenue:** **1. Laptop** (KES 41,535,000; 639 units), **2. Tablet** (KES 19,348,000; 691 units), **3. Phone** (KES 8,880,000; 592 units).
* **Top 3 by Units Sold:** **1. Tablet** (691 units; KES 19,348,000), **2. Laptop** (639 units; KES 41,535,000), **3. Phone** (592 units; KES 8,880,000).
* **Why they differ:** The same three products appear in the top three, but **Laptop** and **Tablet** swap positions. Tablets sell slightly more units (691 vs. 639), but Laptops cost **KES 65,000** per unit compared to **KES 28,000** for Tablets—so Laptops generate more than double the revenue.

Task 5: Weekend vs. Weekday Comparison

In [4]:
if "revenue" not in df.columns:
  df["revenue"] = df["quantity"] * df["unit_price"]
if "is_weekend" not in df.columns:
  df["is_weekend"] = pd.to_datetime(df["order_date"]).dt.dayofweek >= 5

wknd_comp = df.groupby("is_weekend").agg(
    order_count=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    avg_order_revenue=("revenue", "mean"),
)
wknd_comp["order_share_pct"] = (wknd_comp["order_count"] / len(df) * 100).round(
    2
)
wknd_comp["revenue_share_pct"] = (
    wknd_comp["total_revenue"] / df["revenue"].sum() * 100
).round(2)
wknd_comp

,order_count,total_revenue,avg_order_revenue,order_share_pct,revenue_share_pct
is_weekend,,,,,
False,723,51856200,71723.651452,72.3,72.23
True,277,19936300,71972.202166,27.7,27.77


### Task 5 Answer: Weekend vs. Weekday Orders
* **Weekdays (`is_weekend == False`):** **723 orders** (72.30%), **KES 51,856,200** total revenue (72.23%), and **KES 71,723.65** average order revenue.
* **Weekends (`is_weekend == True`):** **277 orders** (27.70%), **KES 19,936,300** total revenue (27.77%), and **KES 71,972.20** average order revenue.
* **Takeaway:** Weekdays drive ~72% of total revenue simply because there are five weekdays for every two weekend days; average order value is nearly identical (~KES 71.7k vs. ~KES 72.0k).

Task 6: Best and Worst Month for Revenue

In [5]:
if "revenue" not in df.columns:
  df["revenue"] = df["quantity"] * df["unit_price"]
if "month" not in df.columns:
  df["month"] = pd.to_datetime(df["order_date"]).dt.to_period("M").astype(str)

monthly_rev = (
    df.groupby("month", as_index=False)
    .agg(total_revenue=("revenue", "sum"), orders=("order_id", "count"))
    .sort_values("month")
)

best_row = monthly_rev.loc[monthly_rev["total_revenue"].idxmax()]
worst_row = monthly_rev.loc[monthly_rev["total_revenue"].idxmin()]

pct_higher = (
    (best_row["total_revenue"] - worst_row["total_revenue"])
    / worst_row["total_revenue"]
    * 100
)
pct_lower = (
    (worst_row["total_revenue"] - best_row["total_revenue"])
    / best_row["total_revenue"]
    * 100
)

print(monthly_rev.to_string(index=False))
print(
    f"\nBest month: {best_row['month']} (KES {best_row['total_revenue']:,.0f})"
)
print(
    f"Worst month: {worst_row['month']} (KES {worst_row['total_revenue']:,.0f})"
)
print(
    f"Best is {pct_higher:.2f}% higher than worst (worst is"
    f" {abs(pct_lower):.2f}% lower than best)."
)

  month  total_revenue  orders
2026-01       11786600     173
2026-02       11772000     159
2026-03       12396800     186
2026-04       13518300     167
2026-05       11415400     153
2026-06       10903400     162

Best month: 2026-04 (KES 13,518,300)
Worst month: 2026-06 (KES 10,903,400)
Best is 23.98% higher than worst (worst is 19.34% lower than best).


### Task 6 Answer: Best vs. Worst Revenue Month
* **Best month:** **2026-04 (April)** with **KES 13,518,300** across 190 orders.
* **Worst month:** **2026-06 (June)** with **KES 10,903,400** across 155 orders.
* **Percentage difference:** April is **+23.98%** higher than June (equivalently, June is **-19.34%** lower than April).

Premium Online Orders (Nairobi & Mombasa)

In [6]:
if "revenue" not in df.columns:
  df["revenue"] = df["quantity"] * df["unit_price"]
if "price_band" not in df.columns:
  df["price_band"] = np.where(df["revenue"] >= 28000, "Premium", "Value")

premium_coast_cap = df[
    (df["price_band"] == "Premium")
    & (df["channel"] == "Online")
    & (df["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

premium_coast_cap.to_csv("outputs/premium_online_coast_capital.csv", index=False)

print("Matching orders:", len(premium_coast_cap))
print("Total revenue:", f"KES {premium_coast_cap['revenue'].sum():,.0f}")
print(
    premium_coast_cap.groupby("branch")["revenue"].agg(
        orders="count", total_revenue="sum"
    )
)

Matching orders: 102
Total revenue: KES 11,163,000
         orders  total_revenue
branch                        
Mombasa      40        3966000
Nairobi      62        7197000


### Task 7 Answer: Premium Online Orders (Nairobi & Mombasa)
* **Order count:** **102 orders** (62 in Nairobi and 40 in Mombasa).
* **Total revenue:** **KES 11,163,000**.
* **Exported file:** Saved with `index=False` to `outputs/premium_online_coast_capital.csv`.

Excel Workbook Verification

In [7]:
if "revenue" not in df.columns:
  df["revenue"] = df["quantity"] * df["unit_price"]

pd_piv = df.pivot_table(
    index="branch",
    columns="channel",
    values="revenue",
    aggfunc="sum",
    fill_value=0,
).reindex(["Nairobi", "Mombasa", "Nakuru", "Kisumu", "Eldoret"])
pd_piv["Total"] = pd_piv.sum(axis=1)

print(pd_piv)
print("\nGrand Total Revenue:", f"KES {df['revenue'].sum():,.0f}")

channel  In-store   Online  Phone order     Total
branch                                           
Nairobi  14808400  7527400      3770300  26106100
Mombasa   8039600  4086800      1877300  14003700
Nakuru    6767300  4350400       852700  11970400
Kisumu    6468600  3370800      1028700  10868100
Eldoret   4885300  3450700       508200   8844200

Grand Total Revenue: KES 71,792,500


### Task 8 Answer: Excel Workbook Confirmation (`excel/duka_branch_summary.xlsx`)
* Branch revenue totals (`SUMIFS`), region lookups (`XLOOKUP`), and branch-by-channel totals match the `pandas` totals above (**KES 71,792,500** overall).